[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/08_data_vis/08_7_Reading_and_Fixing_Charts.ipynb)

# 08.7: Reading and Fixing Charts

Most of the charts you meet will be made by someone else, and you will have to decide whether to trust them. The questions you ask as a reader are the same ones you should ask about your own charts. This notebook gathers those questions from the rest of the module and then works through six charts that have something wrong with them, fixing each one. None of the charts below contains an error in its data or in its code.

This notebook opens in Google Colab. If Colab warns that the notebook was not authored by Google, click **Run anyway**. Your changes are not kept when you close the tab unless you choose **File > Save a copy in Drive**, so do that if you want to keep your work.

## Questions to ask of any chart

**What does it claim?**
- What is the question, and does this kind of chart fit it?
- What does one row or one point represent?

**What does it show?**
- Does a bar axis start at zero? On a line, what range does the axis cover?
- What time range is shown, and what would be different with an earlier start or a later end?
- Are these counts or rates?

**What might it leave out?**
- Does a mean or a box hide the shape of a group, such as two peaks?
- How many rows are behind each bar or box?
- Have any rows been removed, and does the chart say so?
- Would the pattern change if the data were split into groups?

**Can it be read?**
- Is there a title, are the axes labeled and are the units given?
- Are the categories in a meaningful order?
- Is color being asked to do too much?

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

penguins = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/main/data/data_penguins.csv")
flights = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/main/data/data_flights.csv", parse_dates=["date"])
majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
crime = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_state_crime.csv")
crime = crime.rename(columns={"Rates.Violent.All": "Violent_Rate", "Rates.Property.All": "Property_Rate"})
places_2012 = crime[(crime["Year"] == 2012) & (crime["State"] != "United States")]
print(len(penguins), len(flights), len(majors), len(places_2012))

## Chart 1: too many bars in no useful order

In [ ]:
ax = sns.barplot(data=places_2012, x="State", y="Property_Rate")
ax.set_title("Property crime rates, 2012")
ax.set_xlabel("State")
ax.set_ylabel("Property crimes per 100,000 people")
plt.show()

**The problem:** 51 labels pile on top of each other, and alphabetical order has nothing to do with the question, so you cannot tell which places are high or low. **The fix:** keep the rows that matter, sort them, and put the names on the y axis. The title should say that the chart shows only part of the data.

In [ ]:
top_ten = places_2012.sort_values("Property_Rate", ascending=False).head(10)
ax = sns.barplot(data=top_ten, x="Property_Rate", y="State")
ax.set_title("Ten highest property crime rates, 2012")
ax.set_xlabel("Property crimes per 100,000 people")
ax.set_ylabel("")
plt.show()

The District of Columbia is first at 4860.8, well ahead of South Carolina at 3822.2 and Arkansas at 3660.1.

## Chart 2: bars that do not start at zero

In [ ]:
yearly = flights.groupby("year")["passengers"].sum().reset_index()
last_two = yearly[yearly["year"].isin([1959, 1960])]
ax = sns.barplot(data=last_two, x="year", y="passengers")
ax.set_title("Airline passengers soar")
ax.set_xlabel("Year")
ax.set_ylabel("Passengers (thousands)")
ax.set_ylim(5000, 5800)
plt.show()

**The problem:** the 1960 bar looks more than five times as tall as the 1959 bar. The totals are 5714 and 5140 thousand, so 1960 is about 11 percent higher. Only the parts above 5000 are drawn, and the title leans on the false impression. **The fix:** remove `set_ylim()` so that the bars start at zero, and write a title that states what is shown.

In [ ]:
ax = sns.barplot(data=last_two, x="year", y="passengers")
ax.set_title("Airline passengers, 1959 and 1960")
ax.set_xlabel("Year")
ax.set_ylabel("Passengers (thousands)")
plt.show()

The bars now show a modest increase. An increase of about 11 percent in a year is still real growth, and if that is your point, you can say so in words, using the number.

## Chart 3: a line through categories

In [ ]:
share = majors.groupby("Major_category")["Unemployment_rate"].mean().reset_index()
ax = sns.lineplot(data=share, x="Major_category", y="Unemployment_rate")
ax.set_title("Unemployment by category of major")
ax.set_xlabel("Category of major")
ax.set_ylabel("Mean unemployment rate")
plt.show()

**The problem:** the categories have no natural order, so the ups and downs of the line mean nothing, since they come from alphabetical order. The labels also overlap. **The fix:** a sorted horizontal bar chart.

In [ ]:
ax = sns.barplot(data=share.sort_values("Unemployment_rate", ascending=False), x="Unemployment_rate", y="Major_category")
ax.set_title("Unemployment by category of major")
ax.set_xlabel("Mean unemployment rate (0.05 means 5 percent)")
ax.set_ylabel("")
plt.show()
print(share.sort_values("Unemployment_rate", ascending=False).head(2))
print(share.sort_values("Unemployment_rate").head(2))

Now the ranking can be read directly. Social Science is highest at about 0.096, or 9.6 percent, and Physical Sciences is lowest at about 0.047. Since `Unemployment_rate` is a fraction, the x label explains how to read it.

## Chart 4: one number hiding two groups

In [ ]:
island_mass = penguins.groupby("island")["body_mass_g"].mean().reset_index()
ax = sns.barplot(data=island_mass, x="island", y="body_mass_g")
ax.set_title("Mean body mass by island")
ax.set_xlabel("Island")
ax.set_ylabel("Mean body mass (grams)")
plt.show()

**The problem:** nothing is wrong with these bars as far as they go, but the chart invites the conclusion that penguins on Biscoe are heavy, at a mean of about 4716 grams. One number cannot show whether that is true of the penguins there in general. **The fix:** look at the shape of each group.

In [ ]:
grid = sns.displot(data=penguins, x="body_mass_g", col="island", binwidth=250, height=3)
grid.set_axis_labels("Body mass (grams)", "Number of penguins")
grid.set_titles("{col_name}")
plt.show()
print(pd.crosstab(penguins["island"], penguins["species"]))
adelie = penguins[penguins["species"] == "Adelie"]
print(adelie.groupby("island")["body_mass_g"].mean())

The values on Biscoe spread much more widely than on the other islands, from about 2850 to 6300 grams, and the crosstab explains why: Biscoe has 44 Adelie penguins and 124 Gentoo penguins, and Gentoo penguins are much heavier. The last table settles the question. Adelie penguins have nearly the same mean on all three islands, about 3710, 3688 and 3706 grams, so the island itself does not seem to make penguins heavier. The mean for Biscoe blends two different species, and the difference in the bars comes mostly from which species live where.

## Chart 5: a time range chosen for effect

In [ ]:
illinois = crime[crime["State"] == "Illinois"]
since_peak = illinois[illinois["Year"] >= 1991]
ax = sns.lineplot(data=since_peak, x="Year", y="Violent_Rate")
ax.set_title("Violent crime in Illinois has never been lower")
ax.set_xlabel("Year")
ax.set_ylabel("Violent crimes per 100,000 people")
plt.show()

**The problem:** the chart starts at the highest point in the whole history, so it shows only a fall, from 1039.2 to 414.8. The title's claim is true only within the years shown. **The fix:** show all of the years and let the title name them.

In [ ]:
ax = sns.lineplot(data=illinois, x="Year", y="Violent_Rate")
ax.set_title("Violent crime rate in Illinois, 1960-2012")
ax.set_xlabel("Year")
ax.set_ylabel("Violent crimes per 100,000 people")
plt.show()
print(illinois[illinois["Violent_Rate"] < 414.8]["Year"].to_list())

The fall after 1991 is real and large, about 60 percent. But the full chart shows that the rate was lower than its 2012 value in every year from 1960 through 1968, so "has never been lower" is false.

## Chart 6: color asked to do too much

In [ ]:
ax = sns.scatterplot(data=majors, x="ShareWomen", y="Median", hue="Major_category")
ax.set_title("Share of women and median earnings, by major")
ax.set_xlabel("Share of graduates who are women")
ax.set_ylabel("Median earnings (dollars)")
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1), ncol=1, fontsize=7)
plt.show()

**The problem:** 16 colors are too many to tell apart, and the legend makes the reader look back and forth, so you cannot follow any one category. People judge color less accurately than position, and many readers cannot see some color differences at all. **The fix:** decide what the question is, and use color for only that. If the question is about Engineering, make a column that marks it, and use two colors. `style=` also changes the marker shape, so the difference does not depend on color alone.

In [ ]:
marked = majors.copy()
marked["Group"] = "Other majors"
marked.loc[marked["Major_category"] == "Engineering", "Group"] = "Engineering"
ax = sns.scatterplot(data=marked, x="ShareWomen", y="Median", hue="Group", style="Group")
ax.set_title("Share of women and median earnings, by major")
ax.set_xlabel("Share of graduates who are women")
ax.set_ylabel("Median earnings (dollars)")
plt.show()

Now one question can be answered. Engineering majors sit toward the upper left, with higher earnings and a lower share of women than most other majors, and the highest point, at 110000 dollars, is an Engineering major. The chart shows that the two columns are related across majors, but as notebook 08.5 explained, it does not tell us why.

## Saying what a chart shows

A chart is finished once someone else can read it and you have said what it shows. Compare these two descriptions of the full Illinois chart.

- **Weak:** "This chart shows the violent crime rate in Illinois over time. It goes up and then down."
- **Better:** "The violent crime rate in Illinois rose from 365.1 per 100,000 people in 1960 to a peak of 1039.2 in 1991, and then fell by about 60 percent to 414.8 in 2012, which is still above where it began."

The better description names the pattern, gives numbers with units and includes the years. It also says only what the chart supports. It does not say why the rate changed, because this chart cannot tell us that.

## The whole module in one table

| Question | Chart | Watch out for |
|---|---|---|
| How do categories compare? | Bar, sorted, horizontal if labels are long | Axis not starting at zero, counts versus rates, means hiding shape, too many bars |
| What does one numeric column look like? | Histogram | Bin width, a very long tail, columns with few distinct values |
| How does a number differ across groups? | Box plots, one histogram per group, or strip plot | Boxes hiding two peaks, boxes hiding how few rows there are, overlapping histograms |
| Are two numeric columns related? | Scatter plot | Unusual points, curves, groups inside the data, overplotting, correlation mistaken for cause |
| How does something change over time? | Line | Time range, axis range, too many lines, hidden gaps, a line through categories |
| How does a whole divide into parts? | Bar or standardized bar; a pie only for a few parts | Losing group sizes, stacked parts that are hard to compare |

## Practice problems

These problems are not collected or graded. They are here so that you can find out whether you can write this code yourself before the reading check and the pair activity. Write your answer in the cell under each problem, run it, and then run the check cell below it, which tells you whether your answer is right. The check cells are collapsed because they contain the answers, so try the problem before you open one.

Run the next two cells first. The first one creates fresh data for the problems, and the second sets up the checks.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

penguins = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/main/data/data_penguins.csv")
flights = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/main/data/data_flights.csv", parse_dates=["date"])
majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
crime = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_state_crime.csv")
crime = crime.rename(columns={"Rates.Violent.All": "Violent_Rate", "Rates.Property.All": "Property_Rate"})
places_2012 = crime[(crime["Year"] == 2012) & (crime["State"] != "United States")]
ohio_rows = crime[crime["State"] == "Ohio"]
yearly = flights.groupby("year")["passengers"].sum().reset_index()
first_two = yearly[yearly["year"].isin([1949, 1950])]
low_ten_want = places_2012.sort_values("Violent_Rate").head(10)["Violent_Rate"].to_list()
category_pay = majors.groupby("Major_category")["Median"].mean().reset_index()
pay_want = category_pay.sort_values("Median", ascending=False)["Median"].to_list()
count_want = majors["Major_category"].value_counts().to_list()

In [ ]:
#@title Run this cell to set up the checks
import numpy as np
import pandas as pd


def same(got, want):
    """True when got matches want: same labels and values for a series, same value for a number."""
    try:
        if isinstance(want, pd.DataFrame):
            if not isinstance(got, pd.DataFrame) or list(got.columns) != list(want.columns) or len(got) != len(want):
                return False
            if not pd.api.types.is_integer_dtype(want.index) and list(got.index) != list(want.index):
                return False
            got, want = got.reset_index(drop=True), want.reset_index(drop=True)
            for column in want.columns:
                if pd.api.types.is_numeric_dtype(want[column]):
                    if not np.allclose(pd.to_numeric(got[column], errors="coerce").astype(float), want[column].astype(float), equal_nan=True):
                        return False
                elif got[column].fillna("").astype(str).to_list() != want[column].fillna("").astype(str).to_list():
                    return False
            return True
        if isinstance(want, pd.Series):
            if not isinstance(got, pd.Series) or list(got.index) != list(want.index):
                return False
            if pd.api.types.is_numeric_dtype(want):
                return bool(np.allclose(got.astype(float), want.astype(float), equal_nan=True))
            return list(got) == list(want)
        if isinstance(want, (list, np.ndarray)):
            return list(got) == list(want)
        if isinstance(want, str):
            return got == want
        if isinstance(got, (pd.Series, pd.DataFrame, list, np.ndarray)):
            return False
        return bool(np.isclose(float(got), float(want)))
    except (TypeError, ValueError):
        return False


def check(number, name, want, hint, unchanged=None):
    """Print feedback for one problem. unchanged is (variable name, its starting value)."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
    elif not same(globals()[name], want):
        print(f"Problem {number}: not yet. {hint}")
    elif unchanged is not None and not globals()[unchanged[0]].equals(unchanged[1]):
        print(f"Problem {number}: {name} is right, but {unchanged[0]} changed. Work on a copy, then run the setup cell again.")
    else:
        print(f"Problem {number}: correct.")


def check_chart(number, name, test, hint, y_label=True):
    """Check a chart stored in a variable: it has labels, and test(chart) is True."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
        return
    chart = globals()[name]
    if not hasattr(chart, "get_title"):
        print(f"Problem {number}: not yet. Store what the seaborn function returns in {name}.")
    elif not chart.get_title() or not chart.get_xlabel() or (y_label and not chart.get_ylabel()):
        print(f"Problem {number}: not yet. Give the chart a title and label its axes.")
    else:
        try:
            ok = bool(test(chart))
        except (AttributeError, IndexError, TypeError, ValueError):
            ok = False
        print(f"Problem {number}: correct." if ok else f"Problem {number}: not yet. {hint}")

### Problem 1
The cell below already has a chart that shows every place in `places_2012`. Run it. The question is which ten places have the lowest `Violent_Rate`. Fix it so that it shows just those ten, as a horizontal bar chart, from lowest to highest. Change the title to match. Keep the chart stored in `low_ten`, with an x axis label.

In [ ]:
# Problem 1
low_ten = sns.barplot(data=places_2012, x="State", y="Violent_Rate")
low_ten.set_title("Violent crime rates, 2012")
low_ten.set_xlabel("State")
low_ten.set_ylabel("Violent crimes per 100,000 people")
plt.show()

In [ ]:
#@title Check Problem 1
check_chart(1, "low_ten", lambda c: np.allclose([p.get_width() for p in c.patches if p.get_width() > 0], low_ten_want) and "ten" in c.get_title().lower(), "Sort from low to high, take the first ten, put State on y, and say ten in the title.", y_label=False)

### Problem 2
The cell below already has a chart of total passengers in 1949 and 1950, stored in `fair_years`. Fix the problem with its axis, and write a title that states what is shown instead of claiming a boom.

In [ ]:
# Problem 2
fair_years = sns.barplot(data=first_two, x="year", y="passengers")
fair_years.set_title("A boom in air travel")
fair_years.set_xlabel("Year")
fair_years.set_ylabel("Passengers (thousands)")
fair_years.set_ylim(1500, 1700)
plt.show()

In [ ]:
#@title Check Problem 2
check_chart(2, "fair_years", lambda c: c.get_ylim()[0] == 0 and "boom" not in c.get_title().lower(), "Remove the set_ylim() line and change the title.")

### Problem 3
The cell below already has a line through the categories in `category_pay`. Fix it by drawing a horizontal bar chart sorted from highest to lowest. Keep the chart stored in `pay_bars`, with a title and an x axis label.

In [ ]:
# Problem 3
pay_bars = sns.lineplot(data=category_pay, x="Major_category", y="Median")
pay_bars.set_title("Median earnings by category of major")
pay_bars.set_xlabel("Category of major")
pay_bars.set_ylabel("Mean of the median earnings (dollars)")
plt.show()

In [ ]:
#@title Check Problem 3
check_chart(3, "pay_bars", lambda c: np.allclose([p.get_width() for p in c.patches if p.get_width() > 0], pay_want), "Sort category_pay by Median with ascending=False, and use sns.barplot() with Median on x.", y_label=False)

### Problem 4
The cell below already has a pie chart of the number of majors in each category. Replace it with a horizontal count plot sorted from most to fewest, using `order=` with the index of the `category_counts` series. Store the chart in `count_bars`, with a title and an x axis label.

In [ ]:
# Problem 4
category_counts = majors["Major_category"].value_counts()
plt.pie(category_counts.values, labels=category_counts.index)
plt.title("Number of majors in each category")
plt.show()

In [ ]:
#@title Check Problem 4
check_chart(4, "count_bars", lambda c: [p.get_width() for p in c.patches if p.get_width() > 0] == count_want, "Use sns.countplot() with data=majors, y=\"Major_category\" and order=category_counts.index.", y_label=False)

### Problem 5
The cell below already has a chart of `Violent_Rate` in Ohio that starts at Ohio's highest year. Fix it to show all of `ohio_rows`, and have the title name the years 1960 and 2012. Keep the chart stored in `ohio_line`.

In [ ]:
# Problem 5
peak_year = ohio_rows.loc[ohio_rows["Violent_Rate"].idxmax(), "Year"]
since_peak = ohio_rows[ohio_rows["Year"] >= peak_year]
ohio_line = sns.lineplot(data=since_peak, x="Year", y="Violent_Rate")
ohio_line.set_title("Violent crime is falling in Ohio")
ohio_line.set_xlabel("Year")
ohio_line.set_ylabel("Violent crimes per 100,000 people")
plt.show()

In [ ]:
#@title Check Problem 5
check_chart(5, "ohio_line", lambda c: len(c.get_lines()[0].get_xdata()) == 53 and "1960" in c.get_title() and "2012" in c.get_title(), "Use data=ohio_rows with no filter, and put 1960 and 2012 in the title.")

### Problem 6
The mean body mass of penguins is higher on Biscoe than on Dream or Torgersen. Based on the crosstab in Chart 4, what is the most likely explanation? Store your answer in a variable named `biscoe_reason`, as one of `"food on the island"` or `"species"`.

In [ ]:
# Problem 6

In [ ]:
#@title Check Problem 6
check(6, "biscoe_reason", "species", "Look at which species live on each island.")

### Problem 7
Make a scatter plot of `ShareWomen` and `Unemployment_rate` from `majors`, with `hue=` and `style=` set to a two-value column that marks the `Business` category. First create a copy named `marked` with a new column `Group`, and then store the chart in `business_chart`. Give it a title and label both axes.

In [ ]:
# Problem 7

In [ ]:
#@title Check Problem 7
check_chart(7, "business_chart", lambda c: len(c.get_legend().get_texts()) == 2, "Make the Group column with .loc and a mask, and pass hue=\"Group\" and style=\"Group\".")

## What's next

That finishes the notebooks for the course. You can now load a dataset, check it, select from it, summarize it and compare groups, and you can choose a chart that fits your question, draw it, defend it and say what it shows. These are the core skills of working with data. CS/DS 225 builds on them with more detailed work on cleaning, grouping and visualization.